# 10 - Image : Optimisation et Modèles Alternatifs

## Objectif de ce notebook
**Améliorer** la baseline image du notebook 09 en testant : rééquilibrage des classes (class weights), optimisation des hyperparamètres et modèles de type boosting (XGBoost, LightGBM, CatBoost).

**Prérequis** : Exécuter le notebook 09 pour disposer des features image en cache, du label encoder et du modèle baseline dans `models/`.

**Livrable** : Meilleur modèle après optimisation, sauvegardé pour le notebook 11.

---

## Plan
1. Chargement des features et du modèle baseline (notebook 09)
2. Optimisation des hyperparamètres
3. Gestion du déséquilibre (class weights)
4. Modèles avancés (XGBoost, LightGBM, CatBoost)
5. Comparaison complète et sauvegarde du meilleur modèle

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = str(os.cpu_count() or 4)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import pickle
import warnings
warnings.filterwarnings('ignore')

sys.path.append(str(Path('../').resolve()))

from src.modeling import BaselineModels, AdvancedModels
from src.optimization import create_class_weights, random_search_optimization
from src.evaluation import evaluate_model, print_classification_report, plot_confusion_matrix

from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Bibliothèques importées")

In [ ]:
ROOT = Path('../').resolve()
MODELS_DIR = ROOT / 'models'
cache = np.load(MODELS_DIR / 'image_features_cache.npz', allow_pickle=True)
X_features = cache['X']
y_labels = cache['y']

# Superclasse (24 classes) : Publications -> 9999
CODE_SUPERCLASSE = 9999
PUBLICATIONS_CLASSES = {10, 2280, 2403, 2705}

y_labels_superclass = np.array([
    CODE_SUPERCLASSE if y in PUBLICATIONS_CLASSES else y
    for y in y_labels
])

with open(MODELS_DIR / 'label_encoder_image.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

label_encoder_superclass_path = MODELS_DIR / 'label_encoder_image_superclass.pkl'
if label_encoder_superclass_path.exists():
    with open(label_encoder_superclass_path, 'rb') as f:
        label_encoder_superclass = pickle.load(f)
else:
    label_encoder_superclass = LabelEncoder()
    label_encoder_superclass.fit(y_labels_superclass)
    with open(label_encoder_superclass_path, 'wb') as f:
        pickle.dump(label_encoder_superclass, f)

# Encodage labels
y_encoded = label_encoder.transform(y_labels)
y_encoded_sc = label_encoder_superclass.transform(y_labels_superclass)

SCENARIOS = [
    ("27 classes", y_encoded, label_encoder),
    ("24 classes (superclass)", y_encoded_sc, label_encoder_superclass)
]

indices = np.arange(len(y_labels))
train_idx, val_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=y_encoded
)
X_train_split = X_features[train_idx]
X_val_split = X_features[val_idx]

splits_by_scenario = {name: (y_enc[train_idx], y_enc[val_idx]) for name, y_enc, _ in SCENARIOS}

baseline_models_by_scenario = {}
metrics_baseline_by_scenario = {}
y_pred_baseline_by_scenario = {}

for scenario_name, _, _ in SCENARIOS:
    y_train_s, y_val_s = splits_by_scenario[scenario_name]

    scenario_suffix = "superclass" if "superclass" in scenario_name.lower() else "27_classes"
    baseline_files = list(MODELS_DIR.glob(f"image_*_{scenario_suffix}_baseline.pkl"))
    if not baseline_files:
        baseline_files = list(MODELS_DIR.glob('image_*_baseline.pkl'))

    if baseline_files:
        baseline_model = BaselineModels.load_model(baseline_files[0])
        y_pred_baseline = baseline_model.predict(X_val_split)
        metrics_baseline = evaluate_model(y_val_s, y_pred_baseline)
    else:
        baseline_model = LinearSVC(max_iter=2000, random_state=42, dual=False)
        baseline_model.fit(X_train_split, y_train_s)
        y_pred_baseline = baseline_model.predict(X_val_split)
        metrics_baseline = evaluate_model(y_val_s, y_pred_baseline)

    baseline_models_by_scenario[scenario_name] = baseline_model
    metrics_baseline_by_scenario[scenario_name] = metrics_baseline
    y_pred_baseline_by_scenario[scenario_name] = y_pred_baseline

print(f"✅ Données : train {X_train_split.shape[0]:,} | val {X_val_split.shape[0]:,}")
for scenario_name in [s[0] for s in SCENARIOS]:
    print(f"   Baseline F1-macro ({scenario_name}) : {metrics_baseline_by_scenario[scenario_name]['f1_macro']:.4f}")

## 2. Optimisation des hyperparamètres (SVM)

In [ ]:
svm_param_dist = {'C': [0.1, 0.5, 1.0, 2.0, 5.0], 'max_iter': [1000, 2000], 'dual': [False]}
svm_model = LinearSVC(random_state=42, dual=False)

svm_optimized_by_scenario = {}
metrics_svm_opt_by_scenario = {}
y_pred_svm_opt_by_scenario = {}

for scenario_name, _, _ in SCENARIOS:
    y_train_s, y_val_s = splits_by_scenario[scenario_name]
    svm_optimized = random_search_optimization(
        svm_model,
        X_train_split,
        y_train_s,
        svm_param_dist,
        n_iter=15,
        cv=3,
        n_jobs=-1,
        random_state=42
    )
    y_pred_svm_opt = svm_optimized['best_model'].predict(X_val_split)
    metrics_svm_opt = evaluate_model(y_val_s, y_pred_svm_opt)

    svm_optimized_by_scenario[scenario_name] = svm_optimized
    metrics_svm_opt_by_scenario[scenario_name] = metrics_svm_opt
    y_pred_svm_opt_by_scenario[scenario_name] = y_pred_svm_opt

    print(f"SVM optimisé ({scenario_name}) F1-macro : {metrics_svm_opt['f1_macro']:.4f}")

## 3. Class weights

In [ ]:
class_weights_by_scenario = {}
metrics_weighted_by_scenario = {}
y_pred_weighted_by_scenario = {}
svm_weighted_by_scenario = {}

for scenario_name, _, _ in SCENARIOS:
    y_train_s, y_val_s = splits_by_scenario[scenario_name]
    class_weights = create_class_weights(y_train_s, method='balanced')
    svm_weighted = LinearSVC(C=1.0, class_weight=class_weights, random_state=42, dual=False, max_iter=2000)
    svm_weighted.fit(X_train_split, y_train_s)
    y_pred_weighted = svm_weighted.predict(X_val_split)
    metrics_weighted = evaluate_model(y_val_s, y_pred_weighted)

    class_weights_by_scenario[scenario_name] = class_weights
    svm_weighted_by_scenario[scenario_name] = svm_weighted
    metrics_weighted_by_scenario[scenario_name] = metrics_weighted
    y_pred_weighted_by_scenario[scenario_name] = y_pred_weighted

    print(f"SVM + class weights ({scenario_name}) F1-macro : {metrics_weighted['f1_macro']:.4f}")

## 4. Modèles avancés

In [ ]:
advanced_models_by_scenario = {}
trained_advanced_by_scenario = {}

for scenario_name, _, _ in SCENARIOS:
    y_train_s, _ = splits_by_scenario[scenario_name]
    class_weights = class_weights_by_scenario[scenario_name]

    advanced_models = AdvancedModels(random_state=42)
    advanced_models.create_advanced_models(class_weights=class_weights)
    trained_advanced = advanced_models.train_all_models(X_train_split, y_train_s)

    advanced_models_by_scenario[scenario_name] = advanced_models
    trained_advanced_by_scenario[scenario_name] = trained_advanced

print("✅ Modèles avancés entraînés (27 et 24 classes)")

## 5. Comparaison et sauvegarde

In [ ]:
all_results = []

for scenario_name, _, _ in SCENARIOS:
    _, y_val_s = splits_by_scenario[scenario_name]

    all_results.append({
        'Scenario': scenario_name,
        'Model': 'Baseline',
        'F1_macro': metrics_baseline_by_scenario[scenario_name]['f1_macro'],
        'F1_weighted': metrics_baseline_by_scenario[scenario_name]['f1_weighted']
    })
    all_results.append({
        'Scenario': scenario_name,
        'Model': 'SVM Optimized',
        'F1_macro': metrics_svm_opt_by_scenario[scenario_name]['f1_macro'],
        'F1_weighted': metrics_svm_opt_by_scenario[scenario_name]['f1_weighted']
    })
    all_results.append({
        'Scenario': scenario_name,
        'Model': 'SVM Class Weights',
        'F1_macro': metrics_weighted_by_scenario[scenario_name]['f1_macro'],
        'F1_weighted': metrics_weighted_by_scenario[scenario_name]['f1_weighted']
    })

    trained_advanced = trained_advanced_by_scenario[scenario_name]
    advanced_models = advanced_models_by_scenario[scenario_name]
    for name in trained_advanced:
        yp = advanced_models.predict(name, X_val_split)
        m = evaluate_model(y_val_s, yp)
        all_results.append({
            'Scenario': scenario_name,
            'Model': name,
            'F1_macro': m['f1_macro'],
            'F1_weighted': m['f1_weighted']
        })

results_df = pd.DataFrame(all_results).sort_values(['Scenario', 'F1_macro'], ascending=[True, False])
print(results_df.to_string(index=False))

for scenario in results_df['Scenario'].unique():
    print(f"\n🏆 Meilleur ({scenario}) : {results_df[results_df['Scenario'] == scenario].iloc[0]['Model']}")